# Day 10 · 阶段复习与购物车项目

> **前置**: Day01-09 全部(print/input/变量/字符串/分支/循环/函数/列表字典/文件 I/O/异常/OOP/模块/生成器/装饰器)
> **关键问题**: 前 9 天学了"语法点",今天把它们**焊接成一个完整控制台应用**,并通过复习把高频错误集中歼灭,为 NumPy/Pandas 进阶扫清障碍。

**引入:零件已经造好,今天正式造车**

抽问上节: Python 装饰器核心语法是哪两行?(`@functools.wraps(fn)` + `def wrapper(*args, **kwargs)`)。再问: 学了 9 
天,写过"能跑、能存盘、能结算"的东西吗?引出今天的**控制台购物车系统** —— 菜单循环 + 商品浏览 + 加入购物车 + 结算 + JSON 持久化 + 异常防护,一个都不能少。


**1. 三大高频错题回顾**

**错题 1 —— 条件分支**:下段 `elif score >= 80` 永远执行不到,因为 `>= 60` 先吃掉了。

```python
# ❌ 边界错误
if score >= 60:
    print("及格")
elif score >= 80:       # 永远执行不到!
    print("良好")
```

**错题 2 —— 累加器位置**:`total = 0` 写在循环里,每次迭代都被清零,结果只加最后一个。

**错题 3 —— return vs print**:函数 `print` 了结果,`return` 隐式 `None`,其他函数拿不到。排查方法论:**报错不慌张 → 
看最后一行异常类型 → 看报错行号 → `print(type(变量))` + `print(变量)`**。


In [ ]:
# 正确写法示范

# 累加器在循环外初始化
total = 0
for price in [5.5, 12.8, 8.0]:
    total += price
print(f"总价: {total}")         # 26.3

# return 才能真正把值交出去
def calc(prices):
    return sum(prices)

print(calc([5.5, 12.8, 8.0]))   # 26.3

**✏️ 练一练 ⭐⭐**

下面代码有 3 个 Bug,找出来并修复:
```python
def total_price(items):
    for item in items:
        total = 0
        total += item
    print(total)
```

In [ ]:
# ======================
# 学员代码区
# ======================


In [ ]:
# 参考答案
def total_price(items):
    total = 0                     # Bug1:移到循环外
    for item in items:
        total += item
    return total                  # Bug2:return 而不是 print

print(total_price([5.5, 12.8, 8.0]))  # 26.3

**2. 购物车系统 v1 —— 函数封装 + 异常加固 + JSON 持久化**

需求 checklist:商品列表(字典 + 列表) / 菜单循环 / 同商品数量累加 / 结算 + 总价 / 异常加固 / JSON 持久化。

In [ ]:
import json, os

商品库 = [
    {"id": 1, "name": "苹果", "price": 5.50},
    {"id": 2, "name": "牛奶", "price": 12.80},
    {"id": 3, "name": "面包", "price": 8.00},
]
购物车 = []
FILE = "cart.json"

def load_cart():
    if not os.path.exists(FILE):
        return []
    with open(FILE, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []

def save_cart():
    with open(FILE, "w", encoding="utf-8") as f:
        json.dump(购物车, f, ensure_ascii=False, indent=2)

def 查找商品(编号):
    for 商品 in 商品库:
        if 商品["id"] == 编号:
            return 商品
    return None

def 浏览商品():
    for 商品 in 商品库:
        print(f"{商品['id']}. {商品['name']} ￥{商品['price']}")

def 加入购物车():
    try:
        编号 = int(input("请输入商品编号:"))
    except ValueError:
        print("请输入数字!")
        return
    商品 = 查找商品(编号)
    if not 商品:
        print("商品不存在!")
        return
    for 项 in 购物车:
        if 项["id"] == 编号:
            项["qty"] += 1
            save_cart()
            return
    购物车.append({"id": 编号, "name": 商品["name"],
                   "price": 商品["price"], "qty": 1})
    save_cart()

def 结算():
    if not 购物车:
        print("购物车为空!")
        return
    总价 = 0
    for 项 in 购物车:
        小计 = 项["price"] * 项["qty"]
        print(f"{项['name']} x{项['qty']} = ￥{小计:.2f}")
        总价 += 小计
    print(f"总价: ￥{总价:.2f}")
    购物车.clear()
    save_cart()

购物车 = load_cart()

**3. 主循环 `main()` 调度**

口诀:**框架先把函数名填上,每个函数只做一件事,main 只做调度**。

In [ ]:
def main():
    while True:
        print("\n1.浏览商品 2.加入购物车 3.结算 0.退出")
        选 = input("请选择:")
        if 选 == "0":
            print("再见!")
            break
        elif 选 == "1":
            浏览商品()
        elif 选 == "2":
            加入购物车()
        elif 选 == "3":
            结算()
        else:
            print("无效选项!")

if __name__ == "__main__":
    main()

**✏️ 练一练 ⭐⭐⭐**

给购物车加"查看购物车"菜单项(选项 4),打印每条商品名/单价/数量/小计。

In [ ]:
# ======================
# 学员代码区
# ======================


In [ ]:
# 参考答案
def 查看购物车():
    if not 购物车:
        print("购物车为空!")
        return
    for 项 in 购物车:
        小计 = 项["price"] * 项["qty"]
        print(f"{项['name']} ￥{项['price']} x{项['qty']} = ￥{小计:.2f}")

# 在 main() 的菜单里加:
# elif 选 == "4":
#     查看购物车()

**✏️ 练一练 ⭐⭐⭐⭐ —— 综合项目**

把购物车用 OOP 重构:`Product` 类(商品)、`Cart` 类(购物车,含 add/total/show 方法)、`main()` 调度。要求:JSON 
持久化、异常防护、菜单循环一个不少。


In [ ]:
# ======================
# 学员代码区
# ======================


In [ ]:
# 参考答案
import json, os

class Product:
    def __init__(self, pid, name, price):
        self.pid = pid
        self.name = name
        self.price = price

class Cart:
    FILE = "cart_oop.json"

    def __init__(self):
        self.items = []             # [{"product": Product, "qty": int}]
        self._load()

    def add(self, product):
        for item in self.items:
            if item["product"].pid == product.pid:
                item["qty"] += 1
                self._save()
                return
        self.items.append({"product": product, "qty": 1})
        self._save()

    def total(self):
        return sum(
            i["product"].price * i["qty"] for i in self.items
        )

    def show(self):
        for i in self.items:
            p = i["product"]
            print(f"{p.name} ￥{p.price} x{i['qty']}")
        print(f"总价: ￥{self.total():.2f}")

    def _save(self):
        data = [
            {"pid": i["product"].pid, "name": i["product"].name,
             "price": i["product"].price, "qty": i["qty"]}
            for i in self.items
        ]
        with open(self.FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def _load(self):
        if not os.path.exists(self.FILE):
            return
        with open(self.FILE, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                return
        for d in data:
            p = Product(d["pid"], d["name"], d["price"])
            self.items.append({"product": p, "qty": d["qty"]})

**📝 本节试题集**

想更多练习?参见:
- 当堂练:`course/lesson10/in_class/practice01-06.py`(6 道)
- 课后作业:`course/lesson10/homework/task01-03.py`(3 道)

**今日小结**

把 Day01-09 语法焊接成完整项目。三大高频错:条件边界 / 累加器位置 / return vs print。`input` 永远返回字符串,比较必须 `int()` 
转;函数内改列表用 `.clear()` / `append()` 等原地方法,不要赋值 `lst = []`(那会创建新局部变量)。

| 易错点 | 错误写法 | 正确写法 |
|---|---|---|
| 累加器 | 循环内 `total = 0` | 循环外 `total = 0` |
| 条件分支 | `>= 60` 先于 `>= 80` | 从高到低:先 `>= 80` |
| 返回值 | `print(result)` | `return result` |

> 🔴 **易错点**:循环内初始化累加器;`input` 忘 `int()`;`json.load` 读空文件报 `JSONDecodeError`;函数内改列表用原地方法别赋值。